In [1]:
using MAT               # pour charger .mat
using LinearAlgebra
using Distributions
using Random
using Symbolics

In [2]:
@variables θ1 θ2 ω1 ω2 Tm1 Tm2 N           # États
@variables PG1 PG2                          # Entrées connues
@variables KL D1 D2 J1 J2 Ks ω0 a1 a2 b1 b2 Pr1 Pr2 P0      # Paramètres
x = [θ1, θ2, ω1, ω2, Tm1, Tm2, N]
u = [PG1, PG2]

# Sorties mesurées
F12 = KL*(θ1 - θ2)
F21 = KL*(θ2 - θ1)
PL1 = F12 - PG1
PL2 = F21 - PG2
Y = [F12, F21, PG1, PG2, PL1, PL2]

6-element Vector{Num}:
         KL*(θ1 - θ2)
        KL*(-θ1 + θ2)
                  PG1
                  PG2
  -PG1 + KL*(θ1 - θ2)
 -PG2 + KL*(-θ1 + θ2)

In [3]:
ωr = (J1*ω1 + J2*ω2)/(J1+J2)

f = [
    ω1 - ω0,                                           # θ1_dot
    ω2 - ω0,                                           # θ2_dot
    (Tm1 - (PL1 + F12) - D1*(ω1 - ω0))/J1,            # ω1_dot
    (Tm2 - (PL2 + F21) - D2*(ω2 - ω0))/J2,            # ω2_dot
    -a1*(Tm1 - (N*Pr1+P0)*ω1) - b1*(ω1 - ω0), # Tm1_dot
    -a2*(Tm2 - (N*Pr2+P0)*ω2) - b2*(ω2 - ω0), # Tm2_dot
    Ks*((J1*ω1 + J2*ω2)/(J1+J2) - ω0)                                       # N_dot
]

7-element Vector{Num}:
                                          -ω0 + ω1
                                          -ω0 + ω2
  (PG1 + Tm1 - D1*(-ω0 + ω1) - 2KL*(θ1 - θ2)) / J1
 (PG2 + Tm2 - D2*(-ω0 + ω2) - 2KL*(-θ1 + θ2)) / J2
       -b1*(-ω0 + ω1) - (Tm1 - (P0 + N*Pr1)*ω1)*a1
       -b2*(-ω0 + ω2) - (Tm2 - (P0 + N*Pr2)*ω2)*a2
             Ks*((J1*ω1 + J2*ω2) / (J1 + J2) - ω0)

In [4]:
function lie_derivatives(Y, x, f, n)
    Lie = [Y]  # Liste des dérivées
    for k in 1:n
        print(k)
        # Calculer la jacobienne de la dernière dérivée
        J = Symbolics.jacobian(Lie[end], x)
        # Lie suivante
        push!(Lie, simplify.(J * f))
    end
    print(n)
    print(Lie)
    return Lie
end

lie_derivatives (generic function with 1 method)

In [7]:
params = Dict(KL => 3064, D1 =>0.04, D2 =>0.02, J1 =>0.4, J2 =>0.6, Ks =>0.05, ω0 =>314.1592, a1 =>100, a2 =>100, b1 =>2000, b2 =>2000, Pr1 =>100, Pr2 =>50, P0=>400)
f_simpl = substitute(f, params)
Y_simpl = substitute(Y, params)

6-element Vector{SymbolicUtils.BasicSymbolic{Real}}:
 3064(θ1 - θ2)
 3064(-θ1 + θ2)
 PG1
 PG2
 -PG1 + 3064(θ1 - θ2)
 -PG2 + 3064(-θ1 + θ2)

In [ ]:
n_states = length(x)
Lie_list = lie_derivatives(Y, x, f, n_states-1)

# Matrice d'observabilité = empilement vertical des jacobiennes
O = vcat([Symbolics.jacobian(L, x) for L in Lie_list]...)


1234

In [ ]:
# Pour obtenir le rang, on peut substituer des valeurs numériques
# exemple : θ1=0, θ2=0, ω1=ω0, ω2=ω0, etc.
subs_dict = Dict(θ1=>0, θ2=>0, ω1=>ω0, ω2=>ω0, Tm1=>1, Tm2=>1, N=>0, PL1=>1, PL2=>1, KL=>1, D1=>0.1, D2=>0.1, J1=>1, J2=>1, Ks=>0.1)
O_num = substitute.(O, subs_dict)
rank_O = rank(Matrix(O_num))
println("Rang de la matrice d'observabilité : ", rank_O)